# Sémantické Čištění Klíčových Slov (Keyword Cleaning)

Tento notebook stáhne klíčová slova z PostgreSQL databáze, prožene je přes model `paraphrase-multilingual-MiniLM-L12-v2` a spočítá jejich podobnost s tzv. **Seed Listem**.

Před spuštěním si zkontrolujte, že v horním menu **Runtime > Change runtime type** máte nastaveno **Hardware accelerator: T4 GPU**, aby výpočet proběhl bleskově.

In [ ]:
!pip install -q psycopg2-binary sentence-transformers pandas matplotlib

In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

# NASTAVENÍ (Změňte podle potřeby)
SCHEMA = 'pronatal'
TABLE_NAME = 'suggestions'  # suggestions, related_queries, people_also_ask

DB_USER = 'n8n'
DB_PASS = 'n8npass'
DB_HOST = '78.46.190.162'
DB_PORT = '5432'
DB_NAME = 'seo'

conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASS
)
cur = conn.cursor()
cur.execute(f"ALTER TABLE {SCHEMA}.{TABLE_NAME} ADD COLUMN IF NOT EXISTS relevance_score NUMERIC(5,4);")
cur.execute(f"ALTER TABLE {SCHEMA}.{TABLE_NAME} ADD COLUMN IF NOT EXISTS is_relevant BOOLEAN DEFAULT true;")
cur.execute(f"ALTER TABLE {SCHEMA}.{TABLE_NAME} ADD COLUMN IF NOT EXISTS matched_seed TEXT;")
conn.commit()
cur.close()

In [ ]:
print("Načítám seed keywords...")
seeds_df = pd.read_sql_query(f"SELECT DISTINCT keyword FROM {SCHEMA}.seed_keywords WHERE keyword IS NOT NULL", conn)
seed_list = seeds_df['keyword'].tolist()
print(f"Nalezeno {len(seed_list)} seed slov.")

kw_col = 'keyword'
if TABLE_NAME == 'suggestions': kw_col = 'suggestion'
elif TABLE_NAME == 'related_queries': kw_col = 'related_query'
elif TABLE_NAME == 'people_also_ask': kw_col = 'question'

print(f"Načítám data k čištění z {SCHEMA}.{TABLE_NAME}...")
target_df = pd.read_sql_query(f"SELECT {kw_col} AS keyword, relevance_score FROM {SCHEMA}.{TABLE_NAME} WHERE {kw_col} IS NOT NULL", conn)

pending_df = target_df[target_df['relevance_score'].isna()]
pending_kws = pending_df['keyword'].unique().tolist()
print(f"Slov k analýze: {len(pending_kws)} / {len(target_df)}")

In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

if len(pending_kws) > 0:
    batch_size = 512
    seed_emb = model.encode(seed_list, convert_to_tensor=True, batch_size=batch_size)

    results = []
    for i in range(0, len(pending_kws), batch_size):
        batch = pending_kws[i:i+batch_size]
        batch_emb = model.encode(batch, convert_to_tensor=True, batch_size=batch_size)
        
        # Cosine similarity matrix
        cosine_scores = torch.nn.functional.cosine_similarity(
            batch_emb.unsqueeze(1), seed_emb.unsqueeze(0), dim=-1
        )
        max_scores, max_indices = torch.max(cosine_scores, dim=1)
        
        for kw, score, idx in zip(batch, max_scores.cpu().numpy(), max_indices.cpu().numpy()):
            matched_seed = seed_list[idx]
            results.append((float(score), matched_seed, kw))
            
        print(f"Zpracováno {min(i+batch_size, len(pending_kws))} / {len(pending_kws)}")

    # Uložit do DB
    print("Ukládám skóre do databáze...")
    cur = conn.cursor()
    from psycopg2.extras import execute_batch
    execute_batch(cur, f"UPDATE {SCHEMA}.{TABLE_NAME} SET relevance_score = %s, matched_seed = %s WHERE {kw_col} = %s", results, page_size=1000)
    conn.commit()
    cur.close()
else:
    print("Všechna slova už mají skóre spočítané.")

In [ ]:
# Stáhneme všechna skóre pro vizualizaci
df = pd.read_sql_query(f"SELECT {kw_col} AS keyword, relevance_score FROM {SCHEMA}.{TABLE_NAME} WHERE relevance_score IS NOT NULL", conn)

plt.figure(figsize=(10, 5))
plt.hist(df['relevance_score'], bins=50, color='skyblue', edgecolor='black')
plt.title('Distribuce relevance_score')
plt.xlabel('Relevance Score (0.0 = odpad, 1.0 = perfektní shoda)')
plt.ylabel('Počet klíčových slov')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
def print_samples(break_point):
    df = pd.read_sql_query(f"SELECT {kw_col} AS keyword, relevance_score, matched_seed FROM {SCHEMA}.{TABLE_NAME} WHERE relevance_score >= {break_point - 0.01} AND relevance_score <= {break_point + 0.01}", conn)
    print(f"\n--- Vzorky kolem hladiny {break_point} ---")
    for _, row in df.sample(min(10, len(df))).iterrows():
        print(f"[{row["relevance_score"]:.3f}] (seed: {row["matched_seed"]}) {row["keyword"]}")

print_samples(0.60)
print_samples(0.40)
print_samples(0.35)
print_samples(0.30)
print_samples(0.28)
print_samples(0.25)
print_samples(0.20)

In [ ]:
# ZDE NASTAVTE FINÁLNÍ THRESHOLD
THRESHOLD = 0.28  # Vše pod touto hranicí bude zamítnuto (is_relevant = false)

cur = conn.cursor()
cur.execute(f"UPDATE {SCHEMA}.{TABLE_NAME} SET is_relevant = false WHERE relevance_score < %s", (THRESHOLD,))
conn.commit()
removed = cur.rowcount

cur.execute(f"UPDATE {SCHEMA}.{TABLE_NAME} SET is_relevant = true WHERE relevance_score >= %s", (THRESHOLD,))
conn.commit()
kept = cur.rowcount
cur.close()
print(f"✅ Aplikován řez na hladině {THRESHOLD}.")
print(f"Ponecháno: {kept} slov.")
print(f"Odstraněno: {removed} slov.")